In [10]:
# --- Importar librerías ---
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import requests
import zipfile
import io
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from plotnine import *
from sklearn.preprocessing import LabelEncoder
import plotly.express as px
import os
# --- Dataset de ejemplo ---

url = "https://www.kaggle.com/api/v1/datasets/download/philipsanm/sentiment-analysis-in-spanish-tweets"

response = requests.get(url, allow_redirects=True)
if response.status_code == 200:
    print("Download completed ✅")
else:
    print("failed download:", response.status_code)

zip_file = zipfile.ZipFile(io.BytesIO(response.content))

print("Archivos en el zip:", zip_file.namelist())

csv_filename = zip_file.namelist()[0]

df = pd.read_csv(zip_file.open(csv_filename))

pd.set_option('display.max_rows', None)  
pd.set_option('display.max_columns', None)


aliquota = df.sample(n=50, random_state=32)

aliquota.to_csv("aliquota_test.csv", index=False)

df_train = df.drop(aliquota.index).reset_index(drop=True)

print("rows for model:", len(df_train))
print("ficticious new rows ", len(aliquota))

Download completed ✅
Archivos en el zip: ['sentiment_analysis_dataset.csv']
rows for model: 2540
ficticious new rows  50


In [12]:
df

,user,text,date,emotion,sentiment
0,@erreborda,termine bien abrumado después de hoy,"Jan 6, 2024 · 2:53 AM UTC",overwhelmed,scared
1,@shpiderduck,me siento abrumado,"Jan 6, 2024 · 2:35 AM UTC",overwhelmed,scared
2,@Alex_R_art,Me siento un poco abrumado por la cantidad de ...,"Jan 6, 2024 · 12:20 AM UTC",overwhelmed,scared
3,@anggelinaa97,Salvador la única persona que no la ha abrumad...,"Jan 5, 2024 · 10:38 PM UTC",overwhelmed,scared
4,@diegoreyesvqz,Denme un helado o algo que ando full abrumado.,"Jan 5, 2024 · 8:38 PM UTC",overwhelmed,scared
5,@Alfred_Cripto,"Estoy abrumado de airdrops , de youtube y de t...","Jan 5, 2024 · 7:07 PM UTC",overwhelmed,scared
6,@DePacotilla,"#MicroCuento: A veces, sin motivo aparente, o,...","Jan 5, 2024 · 4:39 PM UTC",overwhelmed,scared
7,@messagewvmxn,"Oh, las vacaciones. Tesoros inciertos, venider...","Jan 5, 2024 · 6:19 AM UTC",overwhelmed,scared
8,@my25thour,me siento muy abrumado,"Jan 5, 2024 · 5:30 AM UTC",overwhelmed,scared
9,@JorgeD1428,Consejo que nadie pidió: Si un día te siente...,"Jan 5, 2024 · 4:57 AM UTC",overwhelmed,scared


In [3]:
# --- Dividir datos ---
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.3, random_state=42
)

In [4]:
# --- Vectorizar texto ---
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [6]:
X_train_vec

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11 stored elements and shape (4, 10)>

In [7]:
# --- Entrenar modelo ---
model = LogisticRegression()
model.fit(X_train_vec, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [8]:
# --- Evaluar ---
y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00       2.0

    accuracy                           0.00       2.0
   macro avg       0.00      0.00      0.00       2.0
weighted avg       0.00      0.00      0.00       2.0



c:\Users\alejo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\alejo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\alejo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [9]:
# --- Probar con texto nuevo ---
sample = ["I absolutely love this!", "This was the worst experience ever"]
sample_vec = vectorizer.transform(sample)
print(model.predict(sample_vec))

[0 0]
